In [1]:
import requests
import os
from pathlib import Path
from playwright.async_api import async_playwright

In [2]:
LEXES_API_URL = "https://polylex-admin.epfl.ch/api/v1/lexes?isAbrogated=0"
DATA_DIR = Path("data")

In [3]:
def retrieve_data(api_url):
    response = requests.get(api_url)
    if response.status_code != 200:
        # TODO : a gerer dans les logs
        raise Exception(f"Unexpected status code while fetching : {response.status_code}")
    data = {
        'pdf': {},
        'fedlex': {},
        'others': {}
    }
    for lex in response.json():
        id_lex = lex.get('_id')
        entries = {
            "fr": {
                "type": lex.get("type"),
                "number": lex.get("number"),
                "url_polylex": lex.get('urlFr'),
            },
            "en": {
                "type": lex.get("type"),
                "number": lex.get("number"),
                "url_polylex": lex.get('urlEn'),
            },
        }
        for lang, entry in entries.items():
            url_polylex = entry.get('url_polylex')
            if url_polylex.endswith(".pdf"):
                source_format = "pdf"
            elif "www.admin.ch" in url_polylex or "fedlex.admin.ch" in url_polylex:
                source_format = "fedlex"
            else:
                source_format = "others"
            data[source_format].setdefault(id_lex, {})
            data[source_format][id_lex][lang] = entry
    # TODO : a gerer dans les logs
    total_entries = sum(
        len(langs)
        for source_format in data.values()
        for langs in source_format.values()
    )
    assert(total_entries == 2 * len(response.json()))
    return data

def create_filename(id, lang, type, number, format):
    return lang + "_" + id + "_" + type + "_" + number + f".{format}"

def download_pdf(url, dir, filename):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:147.0) Gecko/20100101 Firefox/147.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(os.path.join(dir, filename), "wb") as f:
            f.write(response.content)
    else:
        print(f"Error: the content for {url} can not be fetched (status: {response.status_code})")

async def resolve_redirects(urls):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        redirects = []
        for url in urls:
            await page.goto(url, wait_until="domcontentloaded")
            await page.wait_for_timeout(1500)
            redirects.append(page.url)
        await browser.close()
        return redirects

def get_fedlex_api_style_url(urls):
    api_urls = []
    for url in urls:
        api_url = url.replace("https://www.fedlex.admin.ch/", "https://fedlex.data.admin.ch/")
        if api_url.endswith("/fr"):
            api_url = api_url.replace("/fr", "")
        api_urls.append(api_url)
    return api_urls

def get_fedlex_pdf(url, lang):
    if lang == "en":
        lang_uri = "ENG"
    else:
        lang_uri = "FRA"
    # TODO : comprendre la requete -> base sur https://fedlex.data.admin.ch/fr-CH/sparql?id=101 puis ChatGPT
    sparql_query = f"""
PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
SELECT ?publicationDate ?dateApplicability ?fileUrl WHERE {{
  ?work jolux:isMemberOf <{url}> ;
        jolux:dateApplicability ?dateApplicability ;
        jolux:isRealizedBy ?expr .
  OPTIONAL {{ ?work jolux:publicationDate ?publicationDate }}
  ?expr jolux:language <http://publications.europa.eu/resource/authority/language/{lang_uri}> ;
        jolux:isEmbodiedBy ?manif .
  ?manif jolux:userFormat <https://fedlex.data.admin.ch/vocabulary/user-format/pdf-a> ;
        jolux:isExemplifiedBy ?fileUrl .
}}
ORDER BY DESC(?publicationDate) DESC(?dateApplicability)
LIMIT 1
""".strip()
    endpoint = "https://fedlex.data.admin.ch/sparqlendpoint"
    r = requests.get(endpoint, params={"query": sparql_query, "format": "application/sparql-results+json"})
    r.raise_for_status()
    data = r.json()
    bindings = data["results"]["bindings"]
    if not bindings:
        print(f"No SPARQL results for {url}")
        return ""
    return bindings[0]["fileUrl"]["value"]

In [4]:
data = retrieve_data(LEXES_API_URL)
os.makedirs(DATA_DIR, exist_ok=True)

# TODO : pdfs viennent de https://www.epfl.ch/about/overview/wp-content/uploads, https://ethrat.stage.mxm.agency/wp-content/uploads, https://ethrat.ch/wp-content/uploads et https://www.edoeb.admin.ch/dam/fr/sd-web/jYpssE3v3PL4/leitfaden_internet-unde-mailueberwachungamarbeitsplatzprivatwirt_FR.pdf
# pdfs
for id_lex, langs in data.get("pdf", {}).items():
    for lang, metadata in langs.items():
        url = metadata.get("url_polylex")
        filename = create_filename(id_lex, lang, metadata.get('type'), metadata.get('number'), "pdf")
        download_pdf(url, DATA_DIR, filename)

# Fedlex
fedlex_jobs = []
for id_lex, langs in data.get("fedlex", {}).items():
    for lang, metadata in langs.items():
        url = metadata.get("url_polylex")
        fedlex_jobs.append((id_lex, lang, url))
fedlex_initial_urls = [u for (_, _, u) in fedlex_jobs]
fedlex_redirected_urls = await resolve_redirects(fedlex_initial_urls)
fedlex_sparql_urls = get_fedlex_api_style_url(fedlex_redirected_urls)
for (id_lex, lang, _), sparql_url in zip(fedlex_jobs, fedlex_sparql_urls):
    data["fedlex"][id_lex][lang]["url_sparql"] = sparql_url
for id_lex, langs in data.get("fedlex", {}).items():
    for lang, metadata in langs.items():
        url = metadata.get("url_sparql")
        filename = create_filename(id_lex, lang, metadata.get('type'), metadata.get('number'), "pdf")
        pdf_url = get_fedlex_pdf(url, lang)
        if pdf_url != "":
            download_pdf(pdf_url, DATA_DIR, filename)
        # TODO : gerer les cas ou on ne recupere pas les pdfs car ko

# others
# TODO (voir ci-dessous)

# TODO : checker que autant de documents que de lexes
#nb_files = sum(1 for f in os.listdir(DATA_DIR))
#assert(nb_files == 2 * len(response.json()))

No SPARQL results for https://fedlex.data.admin.ch/eli/cc/1993/210_210_210/en
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2004/16
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2004/17
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/1967/1505_1553_1547
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2004/179
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2025/782
No SPARQL results for https://fedlex.data.admin.ch/eli/oc/2025/607
No SPARQL results for https://fedlex.data.admin.ch/eli/oc/2025/607
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/1995/3853_3853_3853
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2014/525
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2001/123
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2001/124
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2001/279
No SPARQL results for https://fedlex.data.admin.ch/eli/cc/2012/191
No SPARQL results for https://f

In [5]:
# TODO : comment traiter les 7 cas speciaux (5 car duplications) ?

# https://www.efv.admin.ch/efv/fr/home/themen/finanzpolitik_grundlagen/risiko_versicherungspolitik.html -> site de la confederation avec texte de presentation et pdfs a telecharger
# https://www.epfl.ch/about/overview/fr/reglements-et-directives/compliance/compliance-guide/compliance-guide-2025 -> site de l'epfl avec plein de liens vers d'autres textes (possible de scrapper)
# https://www.epfl.ch/education/phd/fr/reglements -> site de l'epfl avec des liens qui menent vers d'autres liens (fedlex, pdfs, ...)
# http://isa.epfl.ch/imoniteur_ISAP/%21gedpublicreports.htm?ww_i_reportmodel=1715636965 -> pas utile, si ?
# https://www.epfl.ch/education/studies/reglement-et-procedure/plans_etudes -> page avec plein de liens (duplique 3 fois)

#data['others']